# PySpark: CSV → Parquet y comparación de rendimiento

laboratorio:

1. Descargar un CSV real y grande.
2. Cargarlo con PySpark.
3. Definir esquema y explorar calidad de datos.
4. Convertirlo a Parquet.
5. Comparar tiempos de procesamiento entre CSV y Parquet.
6. Analizar planes de ejecución y tamaño en disco.



In [8]:
#!pip install pyspark==4.1.1

In [9]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
import time
from pathlib import Path
import os

spark = (
    SparkSession.builder
    .appName("pyspark CSV-vs-Parquet")
    .config("spark.sql.shuffle.partitions", "16")
    .config("spark.sql.adaptive.enabled", "true")
    .getOrCreate()
)

spark


## 2. dataset CSV

Dataset: **Motor Vehicle Collisions - Crashes** de NYC Open Data.



In [10]:
import urllib.request

DATA_URL = "https://data.cityofnewyork.us/api/views/h9gi-nx95/rows.csv?accessType=DOWNLOAD"
data_dir = Path("data_csv_real")
data_dir.mkdir(parents=True, exist_ok=True)

csv_path = data_dir / "nyc_collisions.csv"

if not csv_path.exists():
    print("Descargando CSV real...")
    urllib.request.urlretrieve(DATA_URL, csv_path)
else:
    print("El archivo ya existe.")

print(csv_path, "MB =", round(csv_path.stat().st_size / (1024 * 1024), 2))


El archivo ya existe.
data_csv_real/nyc_collisions.csv MB = 6.0


## 3. Definición opcional de esquema

In [11]:
schema = T.StructType([
    T.StructField("CRASH DATE", T.StringType(), True),
    T.StructField("CRASH TIME", T.StringType(), True),
    T.StructField("BOROUGH", T.StringType(), True),
    T.StructField("ZIP CODE", T.StringType(), True),
    T.StructField("LATITUDE", T.DoubleType(), True),
    T.StructField("LONGITUDE", T.DoubleType(), True),
    T.StructField("LOCATION", T.StringType(), True),
    T.StructField("ON STREET NAME", T.StringType(), True),
    T.StructField("CROSS STREET NAME", T.StringType(), True),
    T.StructField("OFF STREET NAME", T.StringType(), True),
    T.StructField("NUMBER OF PERSONS INJURED", T.IntegerType(), True),
    T.StructField("NUMBER OF PERSONS KILLED", T.IntegerType(), True),
    T.StructField("NUMBER OF PEDESTRIANS INJURED", T.IntegerType(), True),
    T.StructField("NUMBER OF PEDESTRIANS KILLED", T.IntegerType(), True),
    T.StructField("NUMBER OF CYCLIST INJURED", T.IntegerType(), True),
    T.StructField("NUMBER OF CYCLIST KILLED", T.IntegerType(), True),
    T.StructField("NUMBER OF MOTORIST INJURED", T.IntegerType(), True),
    T.StructField("NUMBER OF MOTORIST KILLED", T.IntegerType(), True),
    T.StructField("CONTRIBUTING FACTOR VEHICLE 1", T.StringType(), True),
    T.StructField("CONTRIBUTING FACTOR VEHICLE 2", T.StringType(), True),
    T.StructField("CONTRIBUTING FACTOR VEHICLE 3", T.StringType(), True),
    T.StructField("CONTRIBUTING FACTOR VEHICLE 4", T.StringType(), True),
    T.StructField("CONTRIBUTING FACTOR VEHICLE 5", T.StringType(), True),
    T.StructField("COLLISION_ID", T.LongType(), True),
    T.StructField("VEHICLE TYPE CODE 1", T.StringType(), True),
    T.StructField("VEHICLE TYPE CODE 2", T.StringType(), True),
    T.StructField("VEHICLE TYPE CODE 3", T.StringType(), True),
    T.StructField("VEHICLE TYPE CODE 4", T.StringType(), True),
    T.StructField("VEHICLE TYPE CODE 5", T.StringType(), True),
])


## 4. Carga el CSV

In [12]:
start = time.time()


df_csv = (
    spark.read
    .option("header", True)
    .schema(schema)
    .csv(str(csv_path))
)

csv_load_time = time.time() - start
print(f"Tiempo de carga CSV: {csv_load_time:.2f} s")
print("Filas:", df_csv.count())
print("Columnas:", len(df_csv.columns))
df_csv.printSchema()
df_csv.show(5, truncate=False)


Tiempo de carga CSV: 0.09 s
Filas: 33327
Columnas: 29
root
 |-- CRASH DATE: string (nullable = true)
 |-- CRASH TIME: string (nullable = true)
 |-- BOROUGH: string (nullable = true)
 |-- ZIP CODE: string (nullable = true)
 |-- LATITUDE: double (nullable = true)
 |-- LONGITUDE: double (nullable = true)
 |-- LOCATION: string (nullable = true)
 |-- ON STREET NAME: string (nullable = true)
 |-- CROSS STREET NAME: string (nullable = true)
 |-- OFF STREET NAME: string (nullable = true)
 |-- NUMBER OF PERSONS INJURED: integer (nullable = true)
 |-- NUMBER OF PERSONS KILLED: integer (nullable = true)
 |-- NUMBER OF PEDESTRIANS INJURED: integer (nullable = true)
 |-- NUMBER OF PEDESTRIANS KILLED: integer (nullable = true)
 |-- NUMBER OF CYCLIST INJURED: integer (nullable = true)
 |-- NUMBER OF CYCLIST KILLED: integer (nullable = true)
 |-- NUMBER OF MOTORIST INJURED: integer (nullable = true)
 |-- NUMBER OF MOTORIST KILLED: integer (nullable = true)
 |-- CONTRIBUTING FACTOR VEHICLE 1: string (n

## 5. Preparación de datos

In [13]:
df_curated = (
    df_csv
    .withColumn("crash_date_parsed", F.to_date(F.col("CRASH DATE"), "MM/dd/yyyy"))
    .withColumn("year", F.year("crash_date_parsed"))
    .withColumn("month", F.month("crash_date_parsed"))
    .withColumn("total_injured", F.coalesce(F.col("NUMBER OF PERSONS INJURED"), F.lit(0)))
    .withColumn("total_killed", F.coalesce(F.col("NUMBER OF PERSONS KILLED"), F.lit(0)))
)

df_curated.select("crash_date_parsed", "year", "month", "BOROUGH", "total_injured", "total_killed").show(5, truncate=False)

+-----------------+----+-----+--------+-------------+------------+
|crash_date_parsed|year|month|BOROUGH |total_injured|total_killed|
+-----------------+----+-----+--------+-------------+------------+
|2021-09-11       |2021|9    |NULL    |2            |0           |
|2022-03-26       |2022|3    |NULL    |1            |0           |
|2023-11-01       |2023|11   |BROOKLYN|1            |0           |
|2022-06-29       |2022|6    |NULL    |0            |0           |
|2022-09-21       |2022|9    |NULL    |0            |0           |
+-----------------+----+-----+--------+-------------+------------+
only showing top 5 rows


In [14]:
df_csv_cleaned = (
    df_csv
    .withColumn("parsed_crash_date_temp", F.to_date(F.col("CRASH DATE"), "MM/dd/yyyy"))
    .filter(F.col("parsed_crash_date_temp").isNotNull())
    .drop("parsed_crash_date_temp")
)

print(f"Original rows: {df_csv.count()}")
print(f"Rows after filtering invalid dates: {df_csv_cleaned.count()}")
df_csv_cleaned.select("CRASH DATE").show(5, truncate=False)

Original rows: 33327
Rows after filtering invalid dates: 33327
+----------+
|CRASH DATE|
+----------+
|09/11/2021|
|03/26/2022|
|11/01/2023|
|06/29/2022|
|09/21/2022|
+----------+
only showing top 5 rows


In [15]:
(
    df_curated
    .groupBy("year", "month")
    .count()
    .orderBy("year", "month")
    .show(20, truncate=False)
)


+----+-----+-----+
|year|month|count|
+----+-----+-----+
|2012|9    |1    |
|2016|4    |1    |
|2019|4    |1    |
|2019|5    |1    |
|2020|1    |1    |
|2020|3    |1    |
|2020|4    |4    |
|2020|5    |8    |
|2020|6    |1    |
|2020|8    |2    |
|2020|9    |3    |
|2020|11   |1    |
|2020|12   |4    |
|2021|1    |5    |
|2021|2    |12   |
|2021|3    |88   |
|2021|4    |5276 |
|2021|5    |9878 |
|2021|6    |1611 |
|2021|7    |180  |
+----+-----+-----+
only showing top 20 rows


## 6. Escritura a Parquet

In [16]:
parquet_path = "lake/parquet/nyc_collisions"

start = time.time()
(
    df_curated
    .repartition("year", "month")
    .write
    .mode("overwrite")
    .partitionBy("year", "month")
    .parquet(parquet_path)
)
parquet_write_time = time.time() - start

print(f"Tiempo de escritura Parquet: {parquet_write_time:.2f} s")
print("Ruta:", parquet_path)


Tiempo de escritura Parquet: 7.05 s
Ruta: lake/parquet/nyc_collisions


In [17]:
df_parquet = spark.read.parquet(parquet_path)
print("Filas Parquet:", df_parquet.count())
df_parquet.printSchema()


Filas Parquet: 33327
root
 |-- CRASH DATE: string (nullable = true)
 |-- CRASH TIME: string (nullable = true)
 |-- BOROUGH: string (nullable = true)
 |-- ZIP CODE: string (nullable = true)
 |-- LATITUDE: double (nullable = true)
 |-- LONGITUDE: double (nullable = true)
 |-- LOCATION: string (nullable = true)
 |-- ON STREET NAME: string (nullable = true)
 |-- CROSS STREET NAME: string (nullable = true)
 |-- OFF STREET NAME: string (nullable = true)
 |-- NUMBER OF PERSONS INJURED: integer (nullable = true)
 |-- NUMBER OF PERSONS KILLED: integer (nullable = true)
 |-- NUMBER OF PEDESTRIANS INJURED: integer (nullable = true)
 |-- NUMBER OF PEDESTRIANS KILLED: integer (nullable = true)
 |-- NUMBER OF CYCLIST INJURED: integer (nullable = true)
 |-- NUMBER OF CYCLIST KILLED: integer (nullable = true)
 |-- NUMBER OF MOTORIST INJURED: integer (nullable = true)
 |-- NUMBER OF MOTORIST KILLED: integer (nullable = true)
 |-- CONTRIBUTING FACTOR VEHICLE 1: string (nullable = true)
 |-- CONTRIBUTING

## 7. Comparación de tamaño en disco

In [18]:
def folder_size_mb(path):
    total = 0
    for p in Path(path).rglob("*"):
        if p.is_file():
            total += p.stat().st_size
    return total / (1024 * 1024)

csv_size_mb = csv_path.stat().st_size / (1024 * 1024)
parquet_size_mb = folder_size_mb(parquet_path)

print("Tamaño CSV (MB):", round(csv_size_mb, 2))
print("Tamaño Parquet (MB):", round(parquet_size_mb, 2))
print("Reducción (%):", round((csv_size_mb - parquet_size_mb) / csv_size_mb * 100, 2) if csv_size_mb > 0 else None)


Tamaño CSV (MB): 9.0
Tamaño Parquet (MB): 2.15
Reducción (%): 76.1


## 8. Benchmark de consultas: CSV vs Parquet

In [19]:
def benchmark(label, action):
    start = time.time()
    result = action()
    elapsed = time.time() - start
    print(f"{label}: {elapsed:.2f} s")
    return result, elapsed


### Consulta 1: agregación por borough

In [20]:
query_csv_1 = lambda: (
    df_csv
    .groupBy("BOROUGH")
    .agg(
        F.count("*").alias("n_crashes"),
        F.sum(F.coalesce(F.col("NUMBER OF PERSONS INJURED"), F.lit(0))).alias("injured")
    )
    .orderBy(F.desc("n_crashes"))
    .show(truncate=False)
)

query_parquet_1 = lambda: (
    df_parquet
    .groupBy("BOROUGH")
    .agg(
        F.count("*").alias("n_crashes"),
        F.sum(F.coalesce(F.col("total_injured"), F.lit(0))).alias("injured")
    )
    .orderBy(F.desc("n_crashes"))
    .show(truncate=False)
)

_, t_csv_1 = benchmark("CSV - Agregación por borough", query_csv_1)
_, t_parquet_1 = benchmark("Parquet - Agregación por borough", query_parquet_1)


+-------------+---------+-------+
|BOROUGH      |n_crashes|injured|
+-------------+---------+-------+
|NULL         |11571    |6100   |
|BROOKLYN     |7498     |3595   |
|QUEENS       |5830     |2572   |
|BRONX        |4022     |1721   |
|MANHATTAN    |3581     |1424   |
|STATEN ISLAND|825      |393    |
+-------------+---------+-------+

CSV - Agregación por borough: 1.34 s
+-------------+---------+-------+
|BOROUGH      |n_crashes|injured|
+-------------+---------+-------+
|NULL         |11571    |6100   |
|BROOKLYN     |7498     |3595   |
|QUEENS       |5830     |2572   |
|BRONX        |4022     |1721   |
|MANHATTAN    |3581     |1424   |
|STATEN ISLAND|825      |393    |
+-------------+---------+-------+

Parquet - Agregación por borough: 1.84 s


### Consulta 2: filtro temporal + selección de pocas columnas

In [21]:
query_csv_2 = lambda: (
    df_curated
    .filter((F.col("year") >= 2023) & (F.col("BOROUGH").isNotNull()))
    .select("BOROUGH", "year", "month", "total_injured", "total_killed")
    .groupBy("BOROUGH", "year", "month")
    .agg(
        F.sum("total_injured").alias("injured"),
        F.sum("total_killed").alias("killed")
    )
    .orderBy("year", "month", "BOROUGH")
    .show(20, truncate=False)
)

query_parquet_2 = lambda: (
    df_parquet
    .filter((F.col("year") >= 2023) & (F.col("BOROUGH").isNotNull()))
    .select("BOROUGH", "year", "month", "total_injured", "total_killed")
    .groupBy("BOROUGH", "year", "month")
    .agg(
        F.sum("total_injured").alias("injured"),
        F.sum("total_killed").alias("killed")
    )
    .orderBy("year", "month", "BOROUGH")
    .show(20, truncate=False)
)

_, t_csv_2 = benchmark("CSV - Filtro y agregación temporal", query_csv_2)
_, t_parquet_2 = benchmark("Parquet - Filtro y agregación temporal", query_parquet_2)


+-------------+----+-----+-------+------+
|BOROUGH      |year|month|injured|killed|
+-------------+----+-----+-------+------+
|BRONX        |2023|1    |11     |0     |
|BROOKLYN     |2023|1    |36     |0     |
|MANHATTAN    |2023|1    |11     |0     |
|QUEENS       |2023|1    |23     |0     |
|STATEN ISLAND|2023|1    |1      |0     |
|BRONX        |2023|4    |0      |0     |
|BROOKLYN     |2023|4    |0      |0     |
|MANHATTAN    |2023|4    |0      |0     |
|QUEENS       |2023|4    |0      |0     |
|BROOKLYN     |2023|11   |1      |0     |
|BROOKLYN     |2023|12   |0      |1     |
|BRONX        |2024|4    |0      |0     |
|STATEN ISLAND|2024|5    |1      |0     |
|BRONX        |2024|9    |0      |0     |
|STATEN ISLAND|2024|10   |1      |0     |
+-------------+----+-----+-------+------+

CSV - Filtro y agregación temporal: 2.02 s
+-------------+----+-----+-------+------+
|BOROUGH      |year|month|injured|killed|
+-------------+----+-----+-------+------+
|BRONX        |2023|1    |11    

### Resumen de tiempos

In [22]:
summary = [
    ("Carga inicial", csv_load_time, None),
    ("Consulta 1", t_csv_1, t_parquet_1),
    ("Consulta 2", t_csv_2, t_parquet_2),
]

for name, t_csv, t_parquet in summary:
    if t_parquet is not None and t_csv:
        improvement = round((t_csv - t_parquet) / t_csv * 100, 2)
        print(f"{name}: CSV={t_csv:.2f}s | Parquet={t_parquet:.2f}s | Mejora={improvement}%")
    else:
        print(f"{name}: CSV={t_csv:.2f}s")


Carga inicial: CSV=0.09s
Consulta 1: CSV=1.34s | Parquet=1.84s | Mejora=-38.17%
Consulta 2: CSV=2.02s | Parquet=0.86s | Mejora=57.29%


## 9. Planes de ejecución

In [23]:
print("PLAN CSV")
(
    df_curated
    .filter((F.col("year") >= 2023) & (F.col("BOROUGH").isNotNull()))
    .select("BOROUGH", "year", "month", "total_injured", "total_killed")
    .groupBy("BOROUGH", "year", "month")
    .agg(F.sum("total_injured").alias("injured"))
    .explain(True)
)


PLAN CSV
== Parsed Logical Plan ==
'Aggregate ['BOROUGH, 'year, 'month], ['BOROUGH, 'year, 'month, 'sum('total_injured) AS injured#883]
+- Project [BOROUGH#278, year#457, month#458, total_injured#459, total_killed#460]
   +- Filter ((year#457 >= 2023) AND isnotnull(BOROUGH#278))
      +- Project [CRASH DATE#276, CRASH TIME#277, BOROUGH#278, ZIP CODE#279, LATITUDE#280, LONGITUDE#281, LOCATION#282, ON STREET NAME#283, CROSS STREET NAME#284, OFF STREET NAME#285, NUMBER OF PERSONS INJURED#286, NUMBER OF PERSONS KILLED#287, NUMBER OF PEDESTRIANS INJURED#288, NUMBER OF PEDESTRIANS KILLED#289, NUMBER OF CYCLIST INJURED#290, NUMBER OF CYCLIST KILLED#291, NUMBER OF MOTORIST INJURED#292, NUMBER OF MOTORIST KILLED#293, CONTRIBUTING FACTOR VEHICLE 1#294, CONTRIBUTING FACTOR VEHICLE 2#295, CONTRIBUTING FACTOR VEHICLE 3#296, CONTRIBUTING FACTOR VEHICLE 4#297, CONTRIBUTING FACTOR VEHICLE 5#298, COLLISION_ID#299L, VEHICLE TYPE CODE 1#300, ... 9 more fields]
         +- Project [CRASH DATE#276, CRASH T

In [24]:
print("PLAN PARQUET")
(
    df_parquet
    .filter((F.col("year") >= 2023) & (F.col("BOROUGH").isNotNull()))
    .select("BOROUGH", "year", "month", "total_injured", "total_killed")
    .groupBy("BOROUGH", "year", "month")
    .agg(F.sum("total_injured").alias("injured"))
    .explain(True)
)


PLAN PARQUET
== Parsed Logical Plan ==
'Aggregate ['BOROUGH, 'year, 'month], ['BOROUGH, 'year, 'month, 'sum('total_injured) AS injured#892]
+- Project [BOROUGH#637, year#667, month#668, total_injured#665, total_killed#666]
   +- Filter ((year#667 >= 2023) AND isnotnull(BOROUGH#637))
      +- Relation [CRASH DATE#635,CRASH TIME#636,BOROUGH#637,ZIP CODE#638,LATITUDE#639,LONGITUDE#640,LOCATION#641,ON STREET NAME#642,CROSS STREET NAME#643,OFF STREET NAME#644,NUMBER OF PERSONS INJURED#645,NUMBER OF PERSONS KILLED#646,NUMBER OF PEDESTRIANS INJURED#647,NUMBER OF PEDESTRIANS KILLED#648,NUMBER OF CYCLIST INJURED#649,NUMBER OF CYCLIST KILLED#650,NUMBER OF MOTORIST INJURED#651,NUMBER OF MOTORIST KILLED#652,CONTRIBUTING FACTOR VEHICLE 1#653,CONTRIBUTING FACTOR VEHICLE 2#654,CONTRIBUTING FACTOR VEHICLE 3#655,CONTRIBUTING FACTOR VEHICLE 4#656,CONTRIBUTING FACTOR VEHICLE 5#657,COLLISION_ID#658L,VEHICLE TYPE CODE 1#659,... 9 more fields] parquet

== Analyzed Logical Plan ==
BOROUGH: string, year: int,